# Permits (Dataset 1)

https://data.cityofnewyork.us/Housing-Development/DOB-NOW-Build-Approved-Permits/rbx6-tga4/about_data (save as permits.csv)

In [44]:
import pandas as pd

df = pd.read_csv("permits.csv", engine="python", on_bad_lines="skip") # skip any rows that have formatting issues

print(df.shape)
df.head()

(1027287, 46)


,Job Filing Number,Work Permit,Sequence Number,Filing Reason,House No,Street Name,Borough,LOT,Bin,Block,...,Permit Status,Tracking Number,Zip Code,Latitude,Longitude,Community Board,Council District,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
0,Permit is no,Permit is not yet issued,NaN,Initial Permit,43-30,PARSONS BOULEVARD,QUEENS,21.0,4117367,5198.0,...,Signed-off,593832799.0,11355.0,40.756514,-73.817023,407.0,20.0,4.051980e+09,861.0,East Flushing
1,M00923249-I1,M00923249-I1-GC,2.0,Renewal Permit Without Changes,836,BROADWAY,MANHATTAN,39.0,1009209,564.0,...,Signed-off,950357329.0,10003.0,40.733707,-73.990994,102.0,2.0,1.005640e+09,61.0,Greenwich Village
2,M01360498-I1,M01360498-I1-GC,1.0,Initial Permit,115,NORFOLK STREET,Manhattan,7503.0,1088708,353.0,...,Permit Issued,915515932.0,10002.0,40.719143,-73.986984,103.0,1.0,1.003538e+09,18.0,Lower East Side
3,M00629731-I1,M00629731-I1-EW-SD,1.0,Initial Permit,300,EAST 83 STREET,MANHATTAN,49.0,1091871,1545.0,...,Signed-off,638913171.0,10028.0,40.775767,-73.952817,108.0,5.0,1.015450e+09,138.0,Upper East Side-Yorkville
4,X01078056-I1,X01078056-I1-GC-CX,2.0,Renewal Permit Without Changes,5550,FIELDSTON ROAD,BRONX,1825.0,2096907,5845.0,...,Signed-off,422350664.0,10471.0,40.903495,-73.900772,208.0,11.0,2.058452e+09,345.0,Riverdale-Spuyten Duyvil


In [45]:
# only keep columns useful for identifying buildings and their permits

df = df[[
    "Bin",
    "BBL",
    "Job Filing Number",
    "Issued Date",
    "Job Description",
    "Work Type",
    "Permit Status"
]]

df.head()

,Bin,BBL,Job Filing Number,Issued Date,Job Description,Work Type,Permit Status
0,4117367,4.051980e+09,Permit is no,NaN,Demolition of 3 story building,Construction Fence,Signed-off
1,1009209,1.005640e+09,M00923249-I1,2024 May 16 12:00:00 AM,EXTERIOR RENOVATION WORK INCLUDING REMOVAL OF ...,General Construction,Signed-off
2,1088708,1.003538e+09,M01360498-I1,2026 May 18 12:00:00 AM,APPLICATION FOR FACADE AND LEAK REPAIRS AS PER...,General Construction,Permit Issued
3,1091871,1.015450e+09,M00629731-I1,2023 May 17 12:00:00 AM,PROVIDE SPRINKLER AND STANDPIPE SYSTEMS FOR PR...,Standpipe,Signed-off
4,2096907,2.058452e+09,X01078056-I1,2026 Apr 21 12:00:00 AM,Garage repairs as per plans; no change to occu...,General Construction,Signed-off


In [46]:
keywords = "boiler|steam" # look for any job descriptions mentioning boilers or steam boilers spec.


heating_df = df[
    df["Job Description"].str.contains(
        keywords, case=False, na=False
    )
].copy()

# see how many matching permits we found
print(heating_df.shape)
heating_df.head()

(24872, 7)


,Bin,BBL,Job Filing Number,Issued Date,Job Description,Work Type,Permit Status
10,3069264,3.027770e+09,B01174537-I1,2025 Apr 17 12:00:00 AM,CONVERSION AND RENOVATION FROM A 3-FAMILY TO A...,Plumbing,Permit Issued
13,5052791,5.036450e+09,S01016639-S3,2026 Mar 20 12:00:00 AM,Remove incinerator and cap existing flue. Inst...,Mechanical Systems,Permit Issued
40,1091798,NaN,M01099682-I1,2025 May 05 12:00:00 AM,Installation of three (3) new condensing boile...,Boiler Equipment,Permit Issued
73,2103117,2.055060e+09,X00743133-I1,2025 Mar 26 12:00:00 AM,ALTERATION TYPE 2 FILED FOR PLUMBING WORK AND ...,Plumbing,Permit Issued
97,4142415,4.065850e+09,Q00987987-I1,2024 Oct 01 12:00:00 AM,"application for plumbing only; cellar, 001, 00...",Plumbing,Permit Issued


In [47]:
# look through the first 20 descriptions to see how boilers/steam are mentioned

for description in heating_df["Job Description"].head(20):
    print(description)
    print()

CONVERSION AND RENOVATION FROM A 3-FAMILY TO A 1-FAMILY RESIDENCE:CELLAR BATH RELOC,NEW BOILER/TANK-1ST FL:1 KITCHEN & 1 BATH RELOC,NEW GAS GRILL AND SINK-2ND FL: 1 KITCHEN REMOVAL,1 BATH RELOC,1 NEW BATH 1 NEW W/D-3RD FL:1 KITCHEN REMOVAL,1 BATH RELOC

Remove incinerator and cap existing flue. Install new flue lining in chimney. Remove existing thermostats throughout building and replace with new. Remove and replace ductwork, oil piping, vent piping and boiler related piping with new where applicable.

Installation of three (3) new condensing boilers and associated pumps and suction diffusers on the 37th Floor (CO)/ M3 (Mktg) and install drain, flow bypass valve, differential pressure sensor and future boiler connections on the Cellar as per plans.

ALTERATION TYPE 2 FILED FOR PLUMBING WORK AND GAS EQUIPMENT IN RECONSTRUCTED 1-FAMILY DWELLING.  INSTALLATION INCLUDES GAS COMBINATION BOILER/HWH DRYER & COOKING APPLIANCE.  INSTALLATION OF SINKS, LAVATORIES, TOILETS, DISH WASHER, WASHING 

In [48]:
# use keywords in each description to figure out the heating system type
#"burner"

def classify_boiler(description):
    description = str(description).lower()

    # check specific boiler types first
    if "steam boiler" in description:
        return "Steam Boiler"
    elif "hot water boiler" in description:
        return "Hot Water Boiler"
    elif "gas boiler" in description or "gas-fired boiler" in description:
        return "Gas Boiler"
    elif "boiler" in description:
        return "Other Boiler" #mentions boiler but doesn't match specific category
    elif "steam" in description:
        return "Steam System"
    else:
        return "Unknown"

heating_df["heating_system"] = heating_df["Job Description"].apply(classify_boiler) #new col

heating_df["heating_system"].value_counts()

heating_system
Other Boiler        19594
Gas Boiler           2883
Steam System         1451
Hot Water Boiler      478
Steam Boiler          466
Name: count, dtype: int64

In [49]:
# estimate how strongly the permit suggests the building has the equipment

def evidence(description):
    description = str(description).lower()

# replacement or installation = high
# removal = low, proposed work = medium
    if "replac" in description:
        return "High"
    elif "remov" in description or "demol" in description:
        return "Low"
    elif "propos" in description:
        return "Medium"
    elif "install" in description:
        return "High"
    else:
        return "Low"

# add evidence strength as another col
heating_df["evidence_strength"] = heating_df["Job Description"].apply(evidence)

heating_df.head()

,Bin,BBL,Job Filing Number,Issued Date,Job Description,Work Type,Permit Status,heating_system,evidence_strength
10,3069264,3.027770e+09,B01174537-I1,2025 Apr 17 12:00:00 AM,CONVERSION AND RENOVATION FROM A 3-FAMILY TO A...,Plumbing,Permit Issued,Other Boiler,Low
13,5052791,5.036450e+09,S01016639-S3,2026 Mar 20 12:00:00 AM,Remove incinerator and cap existing flue. Inst...,Mechanical Systems,Permit Issued,Other Boiler,High
40,1091798,NaN,M01099682-I1,2025 May 05 12:00:00 AM,Installation of three (3) new condensing boile...,Boiler Equipment,Permit Issued,Other Boiler,High
73,2103117,2.055060e+09,X00743133-I1,2025 Mar 26 12:00:00 AM,ALTERATION TYPE 2 FILED FOR PLUMBING WORK AND ...,Plumbing,Permit Issued,Other Boiler,High
97,4142415,4.065850e+09,Q00987987-I1,2024 Oct 01 12:00:00 AM,"application for plumbing only; cellar, 001, 00...",Plumbing,Permit Issued,Other Boiler,High


In [50]:
print(heating_df["heating_system"].value_counts()) # see how many permits we have for each heating system
print(heating_df["evidence_strength"].value_counts()) # see how many were assigned high, medium, or low evidence

heating_system
Other Boiler        19594
Gas Boiler           2883
Steam System         1451
Hot Water Boiler      478
Steam Boiler          466
Name: count, dtype: int64
evidence_strength
High      16957
Low        7321
Medium      594
Name: count, dtype: int64


In [51]:
building_df = heating_df.groupby("Bin").agg(
    heating_systems=("heating_system", "unique"),
    permit_count=("heating_system", "count"),
    evidence_strength=("evidence_strength", "unique")



    ).reset_index()

building_df.head(20)

,Bin,heating_systems,permit_count,evidence_strength
0,1000003,[Other Boiler],2,[High]
1,1000007,[Gas Boiler],1,[High]
2,1000009,[Gas Boiler],1,[Low]
3,1000047,[Steam System],2,[High]
4,1000825,[Other Boiler],1,[Low]
5,1000826,[Steam System],1,[High]
6,1000835,[Other Boiler],2,[High]
7,1000855,[Other Boiler],2,[High]
8,1000856,[Other Boiler],1,[High]
9,1000862,[Other Boiler],1,[Medium]


In [52]:
print("total building count:", len(building_df))

total building count: 11442


In [53]:
# save the building-level results
building_df.to_csv("boiler_buildings.csv", index=False)

# Historical Permits (Dataset 2)

https://data.cityofnewyork.us/Housing-Development/DOB-Permit-Issuance/ipu4-2q9a/about_data (save as historical_permits.csv)

In [54]:
# load the older DOB permit records
df_old = pd.read_csv("historical_permits.csv", low_memory=False)

print(df_old.shape)
df_old.head()

(3991259, 60)


,BOROUGH,Bin #,House #,Street Name,Job #,Job doc. #,Job Type,Self_Cert,Block,Lot,...,Owner’s House State,Owner’s House Zip Code,PERMIT_SI_NO,LATITUDE,LONGITUDE,COUNCIL_DISTRICT,CENSUS_TRACT,BBL,NTA_NAME,DOBRunDate
0,BROOKLYN,3185434,60,BAY 34 ST,340733647,1,A2,Y,06861,00067,...,NY,11214,3765466,40.599043,-73.992006,47.0,300.0,NaN,Bensonhurst East,06/18/2020
1,QUEENS,4623172,117,BEACH 215 STREET,421538731,1,NB,N,16350,00400,...,NY,11697,3765467,40.554860,-73.923824,32.0,91601.0,NaN,Breezy Point-Belle Harbor-Rockaway Park-Broad ...,06/18/2020
2,STATEN ISLAND,5174573,620,RICHMOND TERRACE,510113501,1,NB,N,00062,00056,...,NY,11201,3765795,40.645602,-74.092344,49.0,81.0,NaN,West New Brighton-New Brighton-St. George,06/18/2020
3,BRONX,2010402,1323,CHISHOLM ST,220723885,1,DM,N,02971,00036,...,NY,11205,3765813,40.830992,-73.895698,16.0,153.0,NaN,Crotona Park East,06/18/2020
4,MANHATTAN,1014387,1,PENN PLAZA,123734717,2,A2,Y,00783,00070,...,NY,10019,3623536,40.751098,-73.992926,3.0,101.0,NaN,Midtown-Midtown South,07/29/2020


In [55]:
# keep building IDs, permit information, and boiler-related fields
df_old = df_old[[
    "Bin #",
    "BBL",
    "Job #",
    "Work Type",
    "Permit Status",
    "Permit Subtype",
    "Oil Gas",
    "Issuance Date",
    "Job Start Date"
]]

df_old.head()

,Bin #,BBL,Job #,Work Type,Permit Status,Permit Subtype,Oil Gas,Issuance Date,Job Start Date
0,3185434,NaN,340733647,OT,ISSUED,OT,NaN,06/17/2020,06/17/2020
1,4623172,NaN,421538731,EQ,ISSUED,FN,NaN,06/17/2020,06/06/2019
2,5174573,NaN,510113501,NaN,ISSUED,NaN,NaN,06/17/2020,10/22/2019
3,2010402,NaN,220723885,NaN,ISSUED,NaN,NaN,06/17/2020,06/17/2020
4,1014387,NaN,123734717,MH,ISSUED,MH,NaN,04/22/2019,04/22/2019


In [56]:
# see what types of construction work are listed
df_old["Work Type"].value_counts()

Work Type
OT    1174847
PL     783100
EQ     578231
MH     349131
SP     140187
BL      75451
FP      65031
FB      41362
FS      24371
SD      22946
CC      11769
FA        265
NB          1
Name: count, dtype: int64

In [57]:
# BL is the boiler work type in the historical DOB dataset
boiler_old = df_old[
    df_old["Work Type"].fillna("").str.strip().str.upper() == "BL"
].copy()

print(boiler_old.shape)
boiler_old.head()

(75451, 9)


,Bin #,BBL,Job #,Work Type,Permit Status,Permit Subtype,Oil Gas,Issuance Date,Job Start Date
75,4188214,NaN,421105066,BL,ISSUED,BL,NaN,07/02/2020,07/12/2018
169,3080287,NaN,300736719,BL,ISSUED,BL,NaN,07/01/1998,07/01/1998
203,1055996,NaN,123450568,BL,ISSUED,BL,NaN,01/06/2021,06/13/2019
417,3335262,NaN,321079006,BL,ISSUED,BL,NaN,02/17/2021,12/21/2018
674,4204184,NaN,421614621,BL,ISSUED,BL,NaN,04/08/2021,07/20/2018


In [58]:
# check what additional information the boiler permits provide
print(boiler_old["Permit Subtype"].value_counts(dropna=False))

print(boiler_old["Oil Gas"].value_counts(dropna=False))

print(boiler_old["Permit Status"].value_counts(dropna=False))

Permit Subtype
BL    75451
Name: count, dtype: int64
Oil Gas
NaN    75451
Name: count, dtype: int64
Permit Status
ISSUED        74374
RE-ISSUED       686
IN PROCESS      355
NaN              36
Name: count, dtype: int64


In [59]:
# boiler work type tells us the permit involves boilers
# but not necessarily what type of boiler
boiler_old["heating_system"] = "Other Boiler"

#default to medium evidence strength for the older permits, since they don't have detailed descriptions
boiler_old["evidence_strength"] = "Medium"

boiler_old.head()

,Bin #,BBL,Job #,Work Type,Permit Status,Permit Subtype,Oil Gas,Issuance Date,Job Start Date,heating_system,evidence_strength
75,4188214,NaN,421105066,BL,ISSUED,BL,NaN,07/02/2020,07/12/2018,Other Boiler,Medium
169,3080287,NaN,300736719,BL,ISSUED,BL,NaN,07/01/1998,07/01/1998,Other Boiler,Medium
203,1055996,NaN,123450568,BL,ISSUED,BL,NaN,01/06/2021,06/13/2019,Other Boiler,Medium
417,3335262,NaN,321079006,BL,ISSUED,BL,NaN,02/17/2021,12/21/2018,Other Boiler,Medium
674,4204184,NaN,421614621,BL,ISSUED,BL,NaN,04/08/2021,07/20/2018,Other Boiler,Medium


In [60]:
# check how many boiler permits and buildings we found
print("Total boiler permits:", len(boiler_old))
print("Unique buildings:", boiler_old["Bin #"].nunique())

Total boiler permits: 75451
Unique buildings: 48630


In [61]:
# group permits belonging to the same building
grouped_old = boiler_old.groupby("Bin #")

# get the different heating systems for each building
systems_old = grouped_old["heating_system"].unique()

# get the different evidence levels
evidence_old = grouped_old["evidence_strength"].unique()

# count permits per building
counts_old = grouped_old.size()

building_old = pd.DataFrame({
    "heating_systems": systems_old,
    "evidence_strength": evidence_old,
    "permit_count": counts_old
}).reset_index()

# rename BIN column to match our first dataset
building_old = building_old.rename(columns={"Bin #": "Bin"})

building_old.head(20)

,Bin,heating_systems,evidence_strength,permit_count
0,1000003,[Other Boiler],[Medium],2
1,1000008,[Other Boiler],[Medium],1
2,1000012,[Other Boiler],[Medium],1
3,1000024,[Other Boiler],[Medium],1
4,1000032,[Other Boiler],[Medium],3
5,1000035,[Other Boiler],[Medium],3
6,1000036,[Other Boiler],[Medium],3
7,1000040,[Other Boiler],[Medium],1
8,1000041,[Other Boiler],[Medium],2
9,1000047,[Other Boiler],[Medium],2


In [62]:
# count the total buildings identified from historical permits
print("Total historical buildings:", len(building_old))

Total historical buildings: 48630


In [63]:
# save one row per building
building_old.to_csv("historical_boiler_buildings.csv", index=False)

In [64]:
# make BIN values consistent across both datasets
building_df["Bin"] = building_df["Bin"].astype("string").str.strip()
building_old["Bin"] = building_old["Bin"].astype("string").str.strip()

# remove missing or blank BINs
building_df = building_df[
    building_df["Bin"].notna() & (building_df["Bin"] != "")
]

building_old = building_old[
    building_old["Bin"].notna() & (building_old["Bin"] != "")
]

# Merging the Two Datasets

In [65]:
# rename columns to identify the dataset
building_df = building_df.rename(columns={
    "heating_systems": "dob_now_heating_systems",
    "evidence_strength": "dob_now_evidence",
    "permit_count": "dob_now_permit_count"
})

building_old = building_old.rename(columns={
    "heating_systems": "bis_heating_systems",
    "evidence_strength": "bis_evidence",
    "permit_count": "bis_permit_count"
})

In [66]:
# combine both datasets using the building BIN
merged_df = pd.merge(
    building_df,
    building_old,
    on="Bin",
    how="outer"
)

merged_df.head(20)

,Bin,dob_now_heating_systems,dob_now_permit_count,dob_now_evidence,bis_heating_systems,bis_evidence,bis_permit_count
0,1000003,[Other Boiler],2.0,[High],[Other Boiler],[Medium],2.0
1,1000007,[Gas Boiler],1.0,[High],NaN,NaN,NaN
2,1000008,NaN,NaN,NaN,[Other Boiler],[Medium],1.0
3,1000009,[Gas Boiler],1.0,[Low],NaN,NaN,NaN
4,1000012,NaN,NaN,NaN,[Other Boiler],[Medium],1.0
5,1000024,NaN,NaN,NaN,[Other Boiler],[Medium],1.0
6,1000032,NaN,NaN,NaN,[Other Boiler],[Medium],3.0
7,1000035,NaN,NaN,NaN,[Other Boiler],[Medium],3.0
8,1000036,NaN,NaN,NaN,[Other Boiler],[Medium],3.0
9,1000040,NaN,NaN,NaN,[Other Boiler],[Medium],1.0


In [67]:
# check how many buildings we have after merging
print("Total unique buildings:", len(merged_df))

# count buildings appearing in both datasets
both = merged_df[
    merged_df["dob_now_permit_count"].notna() &
    merged_df["bis_permit_count"].notna()
]

print("Buildings in both datasets:", len(both))

Total unique buildings: 57225
Buildings in both datasets: 2847


In [69]:
# add up the permits from both datasets
merged_df["total_permits"] = (
    merged_df["dob_now_permit_count"].fillna(0) +
    merged_df["bis_permit_count"].fillna(0)
)

merged_df.head(20)

,Bin,dob_now_heating_systems,dob_now_permit_count,dob_now_evidence,bis_heating_systems,bis_evidence,bis_permit_count,total_permits
0,1000003,[Other Boiler],2.0,[High],[Other Boiler],[Medium],2.0,4.0
1,1000007,[Gas Boiler],1.0,[High],NaN,NaN,NaN,1.0
2,1000008,NaN,NaN,NaN,[Other Boiler],[Medium],1.0,1.0
3,1000009,[Gas Boiler],1.0,[Low],NaN,NaN,NaN,1.0
4,1000012,NaN,NaN,NaN,[Other Boiler],[Medium],1.0,1.0
5,1000024,NaN,NaN,NaN,[Other Boiler],[Medium],1.0,1.0
6,1000032,NaN,NaN,NaN,[Other Boiler],[Medium],3.0,3.0
7,1000035,NaN,NaN,NaN,[Other Boiler],[Medium],3.0,3.0
8,1000036,NaN,NaN,NaN,[Other Boiler],[Medium],3.0,3.0
9,1000040,NaN,NaN,NaN,[Other Boiler],[Medium],1.0,1.0


In [70]:
# show whether a building appears in one or both datasets
def get_source(row):
    if pd.notna(row["dob_now_permit_count"]) and pd.notna(row["bis_permit_count"]):
        return "Both"
    elif pd.notna(row["dob_now_permit_count"]):
        return "DOB NOW"
    else:
        return "Historical BIS"

merged_df["source"] = merged_df.apply(get_source, axis=1)

merged_df["source"].value_counts()

source
Historical BIS    45783
DOB NOW            8595
Both               2847
Name: count, dtype: int64

In [71]:
# save one row per building with information from both datasets
merged_df.to_csv("all_boiler_buildings.csv", index=False)

print("Saved", len(merged_df), "buildings")

Saved 57225 buildings
